# 5단계 · 종합 스코어링

**목표:** 3단계에서 모은 지표(기술집약도·수요/공급 부상도)를 **하나의 랭킹**으로 종합하고, **부상도 맵**으로 유망도를 봅니다.

## 방법
- **종합점수** = 세 지표의 평균 (간단·직관적)
- **수요·공급 부상도 맵**: x=공급부상도, y=수요부상도 산점도. **오른쪽 위**일수록 시장·연구가 함께 커지는 유망 신호.

> **1단계에서 확정한 시드를 사용합니다.**
> 데모 시드: **Mathematical finance** (itemId `33589680`, 카테고리 *인공지능*, 지표 *기술집약도*)

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

In [ ]:
# 1단계에서 확정한 시드 (다른 데모 사례로 바꾸려면 값만 교체)
SEED_ID = 33589680          # Mathematical finance
SEED_NAME = "Mathematical finance"
CATEGORY = "인공지능"
DEGREE = 3                  # APOLLO degree=3  →  사용자 체감 '2차수'(이웃의 이웃)

### 지표 수집 (3단계와 동일) 후 종합점수 계산

In [ ]:
res = call_api("GET", f"/open/api/v1/items/{SEED_ID}/network/list",
               params={"degree": DEGREE})
df = pd.DataFrame(res["data"])

# 종합점수 = 3지표 평균 → 정렬
df["종합점수"] = df[["techIntensity", "demandEmergence", "supplyEmergence"]].mean(axis=1).round(2)
df = df.sort_values("종합점수", ascending=False).reset_index(drop=True)
df.insert(0, "순위", range(1, len(df) + 1))
df[["순위", "itemName", "category", "techIntensity",
    "demandEmergence", "supplyEmergence", "종합점수"]].head(15)

### 수요·공급 부상도 맵

In [ ]:
import platform
import matplotlib.pyplot as plt
FONT = "AppleGothic" if platform.system() == "Darwin" else "Malgun Gothic"
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False

top = df.head(20)
plt.figure(figsize=(10, 8))
sc = plt.scatter(top["supplyEmergence"], top["demandEmergence"],
                 s=top["종합점수"] * 6 + 30, c=top["종합점수"], cmap="turbo", alpha=0.8)
for _, r in top.iterrows():
    plt.annotate(str(r["itemName"])[:18], (r["supplyEmergence"], r["demandEmergence"]),
                 fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.colorbar(sc, label="종합점수")
plt.xlabel("공급 부상도"); plt.ylabel("수요 부상도")
plt.title(f"{SEED_NAME} 연관 아이템 · 수요·공급 부상도 맵 (오른쪽 위일수록 유망)")
plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

**정리**
- 종합점수 상위 아이템 = 시드 주변에서 가장 유망한 세부 아이템 후보.
- 이 결과 표를 6단계에서 LLM에 넘겨 **보고서**를 자동 생성합니다.